# Full Pipeline: Continuous Vectorized OLS & Intraday Cointegration
**Strategy**: NSE Intraday Pairs Trading  
**Features**:
1. Minute-by-Minute 7500-Bar Rolling OLS (Vectorized)
2. Intraday Engle-Granger Cointegration (ADF Test on Smooth Spread)
3. Z-Score Mean Reversion Backtest


In [1]:
import os, glob, gc, json, shutil
import sqlite3
import pandas as pd
import numpy as np
from scipy.stats import t as t_dist
from statsmodels.tsa.stattools import adfuller

print("=== /kaggle/input contents ===")
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        fpath = os.path.join(root, f)
        print(f"  {fpath}  ({os.path.getsize(fpath)/(1024**3):.2f} GB)")

hits = glob.glob('/kaggle/input/**/*.sqlite', recursive=True)
if not hits:
    raise FileNotFoundError("No .sqlite found under /kaggle/input")
DB_PATH = hits[0]
print(f"\n✅ DB_PATH = {DB_PATH}")


=== /kaggle/input contents ===
  /kaggle/input/datasets/utkarshpatelthefirst/master-data-1min-db/Master-Data-1min.sqlite  (2.17 GB)

✅ DB_PATH = /kaggle/input/datasets/utkarshpatelthefirst/master-data-1min-db/Master-Data-1min.sqlite


In [2]:
print("=== Loading All DB Data ===")
con = sqlite3.connect(DB_PATH)
df = pd.read_sql("SELECT symbol, timestamp, close FROM ohlcv_1min ORDER BY timestamp", con)
con.close()

df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert('Asia/Kolkata')
time_int = df['dt'].dt.hour * 100 + df['dt'].dt.minute
df_trading = df[(time_int >= 915) & (time_int <= 1529)].copy()
del df; gc.collect()

print("Pivoting to price matrix...")
price_matrix = df_trading.pivot(index='dt', columns='symbol', values='close')
del df_trading; gc.collect()

log_prices = np.log(price_matrix)
print(f"Price matrix: {price_matrix.shape}")


=== Loading All DB Data ===


Pivoting to price matrix...


Price matrix: (44250, 500)


## Stage 1 — Pearson Correlation Screening
**Output**: `pairs_all.csv`


In [ ]:
log_returns = log_prices - log_prices.shift(1)
dates_arr = np.array(price_matrix.index.date)
session_open_mask = np.concatenate([[True], dates_arr[1:] != dates_arr[:-1]])
log_returns.iloc[session_open_mask] = np.nan

print("Computing pairwise Pearson correlation...")
corr_df = log_returns.corr(method='pearson')

symbols = corr_df.columns.tolist()
rows = []
for i in range(len(symbols)):
    for j in range(i + 1, len(symbols)):
        rho = corr_df.iloc[i, j]
        if np.isnan(rho): continue
        n_pair = log_returns[[symbols[i], symbols[j]]].dropna().shape[0]
        if n_pair < 5000: continue
        t_stat = rho * np.sqrt((n_pair - 2) / max(1.0 - rho**2, 1e-12))
        p_val  = 2 * t_dist.sf(abs(t_stat), df=n_pair - 2)
        # REMOVED Pearson p-value filter to test all 1L+ combinations
        rows.append({
            "symbol_a": symbols[i], "symbol_b": symbols[j],
            "pearson_rho": round(rho, 6)
        })

pairs_df = pd.DataFrame(rows).sort_values("pearson_rho", ascending=False).reset_index(drop=True)
pairs_df.to_csv("pairs_all.csv", index=False)
print("Saved Stage 1 outputs.")

TOP_PAIRS = list(zip(pairs_df["symbol_a"], pairs_df["symbol_b"]))
print(f"\nUsing ALL {len(TOP_PAIRS)} Valid Pairs for Execution...")


## Stage 3 — Continuous Vectorized OLS & Execution Engine
**Outputs**: `continuous_ols_production_results_all.csv`, `continuous_ols_top10000.csv`


In [ ]:
ROLLING_WINDOW = 7500
ZSCORE_WINDOW = 7500
Z_ENTRY = 2.0
EOD_EXIT_TIME = 1515
BASE_CAPITAL = 10_000.0
LEVERAGE = 5.0
POS_SIZE = BASE_CAPITAL * LEVERAGE

def calc_zerodha_charges(buy_value, sell_value):
    brok_buy = min(buy_value * 0.0003, 20.0)
    brok_sell = min(sell_value * 0.0003, 20.0)
    stt = sell_value * 0.00025
    etc = (buy_value + sell_value) * 0.0000325
    gst = (brok_buy + brok_sell + etc) * 0.18
    stamp = buy_value * 0.00003
    sebi = (buy_value + sell_value) * 0.000001
    return brok_buy + brok_sell + stt + etc + gst + stamp + sebi

def detect_lagger(ya, yb, timestamps, warmup_bars=7500):
    if len(ya) < warmup_bars + 2: return "a"
    ret_a = np.diff(ya[:warmup_bars])
    ret_b = np.diff(yb[:warmup_bars])
    time_ints = timestamps[:warmup_bars].hour * 100 + timestamps[:warmup_bars].minute
    is_not_915 = (time_ints[1:] != 915)
    ret_a = np.where(is_not_915, ret_a, 0.0)
    ret_b = np.where(is_not_915, ret_b, 0.0)
    c_mat_ab = np.corrcoef(ret_a[1:], ret_b[:-1])
    c_mat_ba = np.corrcoef(ret_b[1:], ret_a[:-1])
    c_ab = c_mat_ab[0, 1] if c_mat_ab.shape == (2, 2) else 0.0
    c_ba = c_mat_ba[0, 1] if c_mat_ba.shape == (2, 2) else 0.0
    c_ab = 0.0 if np.isnan(c_ab) else c_ab
    c_ba = 0.0 if np.isnan(c_ba) else c_ba
    return "b" if abs(c_ba) >= abs(c_ab) else "a"

results_st3 = []

def run_backtest_ols(spread, raw_prices, timestamps, lagger_is_a):
    spread_s = pd.Series(spread)
    roll_mean = spread_s.rolling(ZSCORE_WINDOW).mean()
    roll_std  = spread_s.rolling(ZSCORE_WINDOW).std()
    z_scores  = ((spread_s - roll_mean) / roll_std.replace(0, np.nan)).values
    time_int = timestamps.hour * 100 + timestamps.minute
    is_eod   = (time_int == EOD_EXIT_TIME)
    cash, pos_qty, pos_type, entry_px = BASE_CAPITAL, 0, 0, 0.0
    is_locked_out = False
    trade_log = []
    for t in range(ZSCORE_WINDOW, len(spread)):
        z, price = z_scores[t], raw_prices[t]
        if np.isnan(z) or np.isnan(price) or price <= 0: continue
        
        if is_locked_out and (-1.0 < z < 1.0):
            is_locked_out = False
            
        if pos_qty > 0:
            is_exit = False
            if is_eod[t]: is_exit = True
            elif entry_z_sign >= 1.0 and z <= 0: is_exit = True  # Entered when spread was wide, exited at 0
            elif entry_z_sign <= -1.0 and z >= 0: is_exit = True # Entered when spread was narrow, exited at 0
            
            if is_exit:
                gross = (price - entry_px) * pos_qty if pos_type == 1 else (entry_px - price) * pos_qty
                entry_val = pos_qty * entry_px
                exit_val = pos_qty * price
                buy_val, sell_val = (entry_val, exit_val) if pos_type == 1 else (exit_val, entry_val)
                fees = calc_zerodha_charges(buy_val, sell_val)
                net = gross - fees
                cash += net
                price_diff = abs(price - entry_px) if gross > 0 else -abs(price - entry_px)
                trade_log.append({"gross_pnl": gross, "net_pnl": net, "fees": fees, "price_diff": price_diff, "reason": "EOD" if is_eod[t] else "MEAN_REV"})
                pos_qty, pos_type = 0, 0
                if is_eod[t]: is_locked_out = True
                
        if pos_qty == 0 and not is_eod[t] and not is_locked_out:
            if z <= -Z_ENTRY or z >= Z_ENTRY:
                qty = int(POS_SIZE // price)
                if qty > 0:
                    entry_px, pos_qty = price, qty
                    entry_z_sign = 1.0 if z >= Z_ENTRY else -1.0
                    if z >= Z_ENTRY:
                        pos_type = -1 if lagger_is_a else 1
                    else:
                        pos_type = 1 if lagger_is_a else -1
    
    total_trades = len(trade_log)
    if total_trades > 0:
        gross_pnl = sum(tr["gross_pnl"] for tr in trade_log)
        net_pnl = sum(tr["net_pnl"] for tr in trade_log)
        total_fees = sum(tr["fees"] for tr in trade_log)
        gross_wins = sum(1 for tr in trade_log if tr["gross_pnl"] > 0)
        net_wins = sum(1 for tr in trade_log if tr["net_pnl"] > 0)
        mean_rev_exits = sum(1 for tr in trade_log if tr["reason"] == "MEAN_REV")
        eod_exits = sum(1 for tr in trade_log if tr["reason"] == "EOD")
        avg_price_diff = sum(tr["price_diff"] for tr in trade_log) / total_trades
        avg_fees = total_fees / total_trades
        return {"ols_gross_pnl": gross_pnl, "ols_net_pnl": net_pnl, "total_trades": total_trades, "gross_win_rate": round(gross_wins / total_trades, 4), "net_win_rate": round(net_wins / total_trades, 4), "mean_rev_exits": mean_rev_exits, "eod_exits": eod_exits, "avg_price_captured": round(avg_price_diff, 4), "avg_fee_drag": round(avg_fees, 4)}
    else:
        return {"ols_gross_pnl": 0.0, "ols_net_pnl": 0.0, "total_trades": 0, "gross_win_rate": 0.0, "net_win_rate": 0.0, "mean_rev_exits": 0, "eod_exits": 0, "avg_price_captured": 0.0, "avg_fee_drag": 0.0}

for sym_a, sym_b in TOP_PAIRS:
    df_pair = log_prices[[sym_a, sym_b]].dropna(how='any')
    ya, yb = df_pair[sym_a], df_pair[sym_b]
    times = df_pair.index
    
    lagger_side = detect_lagger(ya.values, yb.values, times, ROLLING_WINDOW)
    lagger_is_a = (lagger_side == "a")
    lagger_sym = sym_a if lagger_is_a else sym_b
    raw_px = price_matrix[lagger_sym].loc[times].values
    
    roll_cov = ya.rolling(window=ROLLING_WINDOW).cov(yb)
    roll_var = yb.rolling(window=ROLLING_WINDOW).var()
    beta = roll_cov / roll_var
    alpha = ya.rolling(window=ROLLING_WINDOW).mean() - beta * yb.rolling(window=ROLLING_WINDOW).mean()
    spread = ya - (alpha + beta * yb)
    
    clean_spread = spread.dropna()
    adf_stat, pval = np.nan, np.nan
    if len(clean_spread) > 100:
        try:
            res = adfuller(clean_spread, maxlag=1)
            adf_stat, pval = res[0], res[1]
        except: pass

    metrics = run_backtest_ols(spread.values, raw_px, times, lagger_is_a)
    
    row = {
        "pair": f"{sym_a}-{sym_b}",
        "lagger_asset": lagger_sym,
        **metrics,
        "adf_stat": round(adf_stat, 4) if not np.isnan(adf_stat) else "",
        "adf_pval": round(pval, 6) if not np.isnan(pval) else "",
    }
    results_st3.append(row)

res_df = pd.DataFrame(results_st3).sort_values("adf_pval", ascending=True)
res_df.to_csv("continuous_ols_production_results_all.csv", index=False)
res_df.head(10000).to_csv("continuous_ols_top10000.csv", index=False)
print("Saved Continuous OLS outputs with Reference Lagger Mirror Logic.")
display(res_df.head(10))


## Publish Output Dataset


In [ ]:
import json, os, shutil
from kaggle.api.kaggle_api_extended import KaggleApi

os.environ['KAGGLE_USERNAME'] = 'utkarshpatelthefirst'
os.environ['KAGGLE_KEY'] = 'fbef16329099428205f671dd5de8337b'

api = KaggleApi()
api.authenticate()

export_dir = '/kaggle/working/dataset_export'
os.makedirs(export_dir, exist_ok=True)

for f in ['pairs_all.csv', 'continuous_ols_production_results_all.csv', 'continuous_ols_top10000.csv']:
    if os.path.exists(f): shutil.copy(f, f'{export_dir}/{f}')

meta = {
    "title"    : "Pairs Continuous OLS Massive Matrix v2",
    "id"       : "utkarshpatelthefirst/pairs-continuous-ols-massive-v2",
    "licenses" : [{"name": "CC0-1.0"}]
}
with open(f'{export_dir}/dataset-metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

try:
    api.dataset_create_new(export_dir, dir_mode='zip', quiet=False)
except Exception as e:
    if "already exists" in str(e).lower() or "409" in str(e):
        print("Dataset exists, updating version...")
        api.dataset_create_version(export_dir, "Update", dir_mode='zip', quiet=False)
    else:
        raise e

print("✅ Dataset ready and published.")
